# Vanishing & Exploding Gradients with PyTorch

This notebook demonstrates vanishing and exploding gradient problems and shows how to fix them.

**Topics covered:**
- What are Vanishing Gradients? (gradients shrink to ~0)
- What are Exploding Gradients? (gradients grow uncontrollably)
- Why they happen: deep networks + bad initialization + wrong activations
- Fixes: **Better Initialization**, **Batch Normalization**, **Gradient Clipping**, **ReLU activations**


## 1. The Problem — Why Gradients Misbehave

During **backpropagation**, gradients are multiplied layer by layer using the chain rule:

```
∂L/∂w₁ = ∂L/∂wₙ × ∂wₙ/∂wₙ₋₁ × ... × ∂w₂/∂w₁
```

In a deep network with many layers:
- If each gradient term < 1 → multiplying many of them → **value collapses to ~0** → **Vanishing Gradient**
- If each gradient term > 1 → multiplying many of them → **value explodes** → **Exploding Gradient**

| Problem | Symptom | Common cause |
|---------|---------|-------------|
| **Vanishing** | Early layers stop learning (gradients ≈ 0) | Sigmoid/Tanh activation + deep network |
| **Exploding** | Loss becomes NaN, weights go to ±∞ | Poor initialization + high learning rate |


## Setup


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

NUM_EPOCHS    = 10
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {DEVICE}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  BATCH_SIZE, shuffle=False)

print(f"Training samples : {len(train_dataset):,}")
print(f"Testing samples  : {len(test_dataset):,}")


## Helper Functions


In [ ]:
loss_function = nn.CrossEntropyLoss()


def train_one_epoch(model, device, loader, optimizer, epoch, clip_grad=None):
    """Train for one epoch. Returns (avg_loss, accuracy, gradient_norms_per_layer)."""
    model.train()
    total_loss, correct = 0, 0
    grad_norms = []   # track gradient norms to visualize vanishing/exploding

    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = loss_function(outputs, target)
        loss.backward()

        # Optionally clip gradients (fix for exploding)
        if clip_grad is not None:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)

        # Record gradient norms for the first batch only (for efficiency)
        if batch_idx == 0:
            norms = []
            for name, param in model.named_parameters():
                if param.grad is not None and 'weight' in name:
                    norms.append(param.grad.norm().item())
            grad_norms.append(norms)

        optimizer.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(dim=1) == target).sum().item()

        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch {epoch} | Step {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader), 100.0 * correct / len(loader.dataset), grad_norms


def evaluate_model(model, device, loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += loss_function(output, target).item()
            correct    += (output.argmax(dim=1) == target).sum().item()
    return total_loss / len(loader), 100.0 * correct / len(loader.dataset)


def plot_curves(train_losses, test_losses, train_accs, test_accs, title_prefix=""):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label="Train Loss", color="blue")
    plt.plot(test_losses,  label="Test Loss",  color="red")
    plt.title(f"{title_prefix} — Loss Curves")
    plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label="Train Acc", color="blue")
    plt.plot(test_accs,  label="Test Acc",  color="red")
    plt.title(f"{title_prefix} — Accuracy Curves")
    plt.xlabel("Epochs"); plt.ylabel("Accuracy %"); plt.legend()

    plt.tight_layout(); plt.show()


def plot_gradient_norms(all_norms_dict):
    """Bar chart comparing gradient norms per layer across models."""
    plt.figure(figsize=(12, 5))
    colors = ["red", "blue", "green", "orange"]
    for i, (label, norms) in enumerate(all_norms_dict.items()):
        if norms:
            flat = norms[0]   # first epoch, first batch
            plt.bar(
                np.arange(len(flat)) + i * 0.2,
                flat,
                width=0.2,
                label=label,
                color=colors[i % len(colors)],
                alpha=0.8
            )
    plt.yscale("log")
    plt.title("Gradient Norms per Layer (log scale) — Epoch 1, Batch 1")
    plt.xlabel("Layer index"); plt.ylabel("Gradient norm (log)")
    plt.legend(); plt.tight_layout(); plt.show()


## Part 1 — Vanishing Gradient (the problem)

**What happens:** Sigmoid activation squashes inputs to (0, 1). Its derivative is at most 0.25.  
In a 10-layer deep network, gradients get multiplied by 0.25 ten times → 0.25¹⁰ ≈ 0.000001.  
Early layers receive almost zero gradient → **they stop learning**.

We force this by using:
- A deep network (10 hidden layers)
- Sigmoid activation everywhere
- Random (non-optimal) weight initialization


In [ ]:
class VanishingGradientModel(nn.Module):
    """Deep network with Sigmoid activations → vanishing gradients."""
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        layers = [nn.Linear(28*28, 128), nn.Sigmoid()]
        for _ in range(8):                           # 8 more hidden layers
            layers += [nn.Linear(128, 128), nn.Sigmoid()]
        layers += [nn.Linear(128, 10)]
        self.net = nn.Sequential(*layers)

        # Bad initialization: all weights set to a small constant
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.constant_(m.weight, 0.1)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(self.flatten(x))


vanish_model     = VanishingGradientModel().to(DEVICE)
vanish_optimizer = optim.SGD(vanish_model.parameters(), lr=0.01)

v_train_losses, v_test_losses = [], []
v_train_accs,   v_test_accs   = [], []
v_grad_norms = []

print("Training model with vanishing gradients...")
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, gnorms = train_one_epoch(vanish_model, DEVICE, train_loader, vanish_optimizer, epoch)
    te_loss, te_acc         = evaluate_model(vanish_model, DEVICE, test_loader)
    v_train_losses.append(tr_loss); v_test_losses.append(te_loss)
    v_train_accs.append(tr_acc);    v_test_accs.append(te_acc)
    if gnorms: v_grad_norms = gnorms
    print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")

plot_curves(v_train_losses, v_test_losses, v_train_accs, v_test_accs, "Vanishing Gradient")


**Analysis:**  
The model barely learns — accuracy stays near random (10%) for many epochs.  
The deep Sigmoid chain crushes gradients before they reach the early layers.  
Notice how the loss curve barely moves — the optimizer gets no useful signal.


## Part 2 — Exploding Gradient (the problem)

**What happens:** Large initial weights + no normalization → activations blow up layer by layer.  
Gradients grow exponentially during backprop → weights shoot to ±∞ → loss becomes **NaN**.

We force this by:
- Large weight initialization (std=2.0)
- High learning rate
- No normalization


In [ ]:
class ExplodingGradientModel(nn.Module):
    """Deep network with large random init → exploding gradients."""
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        layers = [nn.Linear(28*28, 128), nn.Tanh()]
        for _ in range(6):
            layers += [nn.Linear(128, 128), nn.Tanh()]
        layers += [nn.Linear(128, 10)]
        self.net = nn.Sequential(*layers)

        # Bad initialization: very large weights → activations explode
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=2.0)   # ← large std!
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(self.flatten(x))


explode_model     = ExplodingGradientModel().to(DEVICE)
explode_optimizer = optim.SGD(explode_model.parameters(), lr=0.1)   # ← high lr

e_train_losses, e_test_losses = [], []
e_train_accs,   e_test_accs   = [], []
e_grad_norms = []

print("Training model with exploding gradients...")
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, gnorms = train_one_epoch(explode_model, DEVICE, train_loader, explode_optimizer, epoch)
    te_loss, te_acc         = evaluate_model(explode_model, DEVICE, test_loader)
    e_train_losses.append(tr_loss); e_test_losses.append(te_loss)
    e_train_accs.append(tr_acc);    e_test_accs.append(te_acc)
    if gnorms: e_grad_norms = gnorms
    print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")
    if np.isnan(tr_loss):
        print("  ⚠ Loss is NaN — exploding gradient confirmed! Stopping early.")
        break

plot_curves(e_train_losses, e_test_losses, e_train_accs, e_test_accs, "Exploding Gradient")


## Part 3 — Fix 1: Better Weight Initialization

**The idea:** Initialize weights so that the variance of activations stays roughly the same across layers.  
This prevents both shrinking and growing.

| Initialization | Best with | Formula |
|---------------|-----------|----------|
| **Xavier / Glorot** | Sigmoid, Tanh | std = √(2 / (fan_in + fan_out)) |
| **He / Kaiming** | ReLU | std = √(2 / fan_in) |

PyTorch uses He init by default for layers followed by ReLU.


In [ ]:
class GoodInitModel(nn.Module):
    """Same deep network but with proper He initialization + ReLU."""
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        layers = [nn.Linear(28*28, 128), nn.ReLU()]
        for _ in range(8):
            layers += [nn.Linear(128, 128), nn.ReLU()]
        layers += [nn.Linear(128, 10)]
        self.net = nn.Sequential(*layers)

        # He initialization — correct for ReLU
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')   # ← He init
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(self.flatten(x))


init_model     = GoodInitModel().to(DEVICE)
init_optimizer = optim.SGD(init_model.parameters(), lr=0.01)

i_train_losses, i_test_losses = [], []
i_train_accs,   i_test_accs   = [], []
i_grad_norms = []

print("Training with He initialization + ReLU...")
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, gnorms = train_one_epoch(init_model, DEVICE, train_loader, init_optimizer, epoch)
    te_loss, te_acc         = evaluate_model(init_model, DEVICE, test_loader)
    i_train_losses.append(tr_loss); i_test_losses.append(te_loss)
    i_train_accs.append(tr_acc);    i_test_accs.append(te_acc)
    if gnorms: i_grad_norms = gnorms
    print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")

plot_curves(i_train_losses, i_test_losses, i_train_accs, i_test_accs, "He Init + ReLU")


## Part 4 — Fix 2: Batch Normalization

**The idea:** After each linear layer, normalize the activations to have mean≈0 and std≈1.  
This keeps activations in a healthy range across layers, stabilizing the gradient flow.

Benefits:
- Reduces internal covariate shift
- Allows higher learning rates
- Acts as a mild regularizer
- Reduces sensitivity to weight initialization


In [ ]:
class BatchNormModel(nn.Module):
    """Deep network with Batch Normalization after every hidden layer."""
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        layers = [nn.Linear(28*28, 128), nn.BatchNorm1d(128), nn.ReLU()]   # ← BN here
        for _ in range(8):
            layers += [nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU()]
        layers += [nn.Linear(128, 10)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(self.flatten(x))


bn_model     = BatchNormModel().to(DEVICE)
bn_optimizer = optim.Adam(bn_model.parameters(), lr=0.001)

bn_train_losses, bn_test_losses = [], []
bn_train_accs,   bn_test_accs   = [], []
bn_grad_norms = []

print("Training with Batch Normalization...")
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, gnorms = train_one_epoch(bn_model, DEVICE, train_loader, bn_optimizer, epoch)
    te_loss, te_acc         = evaluate_model(bn_model, DEVICE, test_loader)
    bn_train_losses.append(tr_loss); bn_test_losses.append(te_loss)
    bn_train_accs.append(tr_acc);    bn_test_accs.append(te_acc)
    if gnorms: bn_grad_norms = gnorms
    print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")

plot_curves(bn_train_losses, bn_test_losses, bn_train_accs, bn_test_accs, "Batch Normalization")


## Part 5 — Fix 3: Gradient Clipping

**The idea:** Cap the gradient norm to a maximum value before the optimizer step.  
If the total gradient norm exceeds `max_norm`, rescale all gradients proportionally.

```python
nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

This is the standard fix for **exploding gradients** in RNNs and deep networks.  
It doesn't fix vanishing, but it prevents the loss from becoming NaN.


In [ ]:
# Reuse the exploding-gradient architecture but add clipping
class ExplodingGradientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        layers = [nn.Linear(28*28, 128), nn.Tanh()]
        for _ in range(6):
            layers += [nn.Linear(128, 128), nn.Tanh()]
        layers += [nn.Linear(128, 10)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=2.0)
                nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(self.flatten(x))


clip_model     = ExplodingGradientModel().to(DEVICE)
clip_optimizer = optim.SGD(clip_model.parameters(), lr=0.1)

cl_train_losses, cl_test_losses = [], []
cl_train_accs,   cl_test_accs   = [], []
cl_grad_norms = []

print("Training with Gradient Clipping (same bad init, but clipped at 1.0)...")
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, gnorms = train_one_epoch(
        clip_model, DEVICE, train_loader, clip_optimizer, epoch,
        clip_grad=1.0   # ← clipping here
    )
    te_loss, te_acc = evaluate_model(clip_model, DEVICE, test_loader)
    cl_train_losses.append(tr_loss); cl_test_losses.append(te_loss)
    cl_train_accs.append(tr_acc);    cl_test_accs.append(te_acc)
    if gnorms: cl_grad_norms = gnorms
    print(f"→ Epoch {epoch}: Train Acc={tr_acc:.2f}% | Test Acc={te_acc:.2f}%")

plot_curves(cl_train_losses, cl_test_losses, cl_train_accs, cl_test_accs, "Gradient Clipping")


## Part 6 — Gradient Norms Visualization

Here we visualize the actual gradient magnitudes layer by layer to see the problem directly.


In [ ]:
plot_gradient_norms({
    "Vanishing (Sigmoid)": v_grad_norms,
    "He Init + ReLU":      i_grad_norms,
    "Batch Norm":          bn_grad_norms,
    "Clipped":             cl_grad_norms,
})

print("\nReading the chart:")
print("  • Vanishing → gradient norms shrink toward 0 for early layers")
print("  • He + ReLU / BatchNorm → gradient norms stay healthy across all layers")
print("  • Clipping  → norms are bounded, never explode")


## Part 7 — Final Comparison


In [ ]:
plt.figure(figsize=(14, 6))

# Test accuracy comparison
plt.subplot(1, 2, 1)
plt.plot(v_test_accs,  label="Vanishing (Sigmoid+bad init)", color="red",    linestyle="--")
plt.plot(e_test_accs,  label="Exploding (large init)",       color="orange",  linestyle="--")
plt.plot(i_test_accs,  label="Fix: He Init + ReLU",          color="blue")
plt.plot(bn_test_accs, label="Fix: Batch Norm",              color="green")
plt.plot(cl_test_accs, label="Fix: Grad Clipping",           color="purple")
plt.title("Test Accuracy Comparison")
plt.xlabel("Epochs"); plt.ylabel("Accuracy %"); plt.legend(fontsize=8)

# Train loss comparison
plt.subplot(1, 2, 2)
plt.plot(v_train_losses,  label="Vanishing",     color="red",    linestyle="--")
plt.plot(e_train_losses,  label="Exploding",     color="orange",  linestyle="--")
plt.plot(i_train_losses,  label="He Init+ReLU",  color="blue")
plt.plot(bn_train_losses, label="Batch Norm",    color="green")
plt.plot(cl_train_losses, label="Grad Clipping", color="purple")
plt.title("Train Loss Comparison")
plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.legend(fontsize=8)

plt.tight_layout(); plt.show()

print("\nKey takeaways:")
print("  • Vanishing gradient → model barely learns (stuck near random accuracy)")
print("  • Exploding gradient → loss becomes NaN, training collapses")
print("  • He init + ReLU    → stable training without any extra overhead")
print("  • BatchNorm          → fastest convergence, most stable gradients")
print("  • Grad Clipping      → prevents NaN but doesn't fully fix bad init")
print("\nIn practice: use He init + ReLU + BatchNorm together for deep networks.")
